In [0]:
%pip install pypdf

In [0]:
dbutils.library.restartPython()

In [0]:
from pypdf import PdfReader
from datetime import datetime, timezone
from pyspark.sql import Row

VOLUME = "/Volumes/workspace/privacylens/raw_docs"
DOCS = {"gdpr.pdf": "GDPR", "dpdp_act_2023.pdf": "DPDP"}

ingested_at = datetime.now(timezone.utc)
rows = []

for fname, law in DOCS.items():
    reader = PdfReader(f"{VOLUME}/{fname}")
    for page_num, page in enumerate(reader.pages, start=1):
        rows.append(Row(
            law=law,
            source_file=fname,
            page_num=page_num,
            text=page.extract_text() or "",
            ingested_at=ingested_at,
        ))

df = spark.createDataFrame(rows)
df.write.mode("overwrite").saveAsTable("workspace.privacylens.bronze_docs")

print(f"Wrote {df.count()} pages")

In [0]:
spark.sql("""
COMMENT ON TABLE workspace.privacylens.bronze_docs IS
'Bronze: raw page-level text extracted from GDPR and DPDP Act 2023 PDFs in raw_docs volume'
""")

In [0]:
display(spark.sql("""
SELECT law, COUNT(*) AS pages, SUM(LENGTH(text)) AS total_chars
FROM workspace.privacylens.bronze_docs
GROUP BY law
"""))

In [0]:

display(spark.sql("""
SELECT law, page_num, LEFT(text, 300) AS preview
FROM workspace.privacylens.bronze_docs
WHERE law = 'DPDP' AND page_num IN (2, 3)
"""))